# QDIST v4.0.0 final scientific adjudication and freeze preparation

This notebook finalizes the accepted cohort candidate without recomputing feature extraction.

In [ ]:
from pathlib import Path
import json, os, sys

RUN_PACKAGE_TESTS = True
RUN_FINALIZATION = True
PROJECT_ROOT_OVERRIDE = None
SCIENTIFIC_REVIEW_DECISION = "PENDING"
SCIENTIFIC_REVIEWER = "Nevena Musikic"
PUBLISH_AND_FREEZE = False

if PUBLISH_AND_FREEZE:
    raise RuntimeError("Publication/freezing is performed only by the separate atomic freeze scripts.")

PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve() if PROJECT_ROOT_OVERRIDE else Path.cwd().resolve()
while not (PROJECT_ROOT / "src reviewed").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root:", PROJECT_ROOT)
print("Decision:", SCIENTIFIC_REVIEW_DECISION)

In [ ]:
if RUN_PACKAGE_TESTS:
    import subprocess
    tests = [
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400.py",
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400_notebook.py",
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400_cohort.py",
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400_final.py",
    ]
    result = subprocess.run([str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"), "-m", "pytest", *map(str, tests), "-q", "--disable-warnings"], cwd=PROJECT_ROOT)
    if result.returncode:
        raise RuntimeError("Reviewed QDIST tests failed")

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src reviewed"))
from paper1_qc_reviewed.qdist_v400_final import finalize_candidate

SOURCE_ROOT = PROJECT_ROOT / "outputs reviewed" / "nonlinear_distortion" / "qdist-v4.0.0-candidate"
FINAL_ROOT = PROJECT_ROOT / "outputs reviewed" / "nonlinear_distortion" / "qdist-v4.0.0"
PROVENANCE = PROJECT_ROOT / "notebooks reviewed" / "05_QDIST"

required_provenance = [
    PROVENANCE / "QDIST_v400_FINAL_SCIENTIFIC_AUDIT.md",
    PROVENANCE / "QDIST_v400_FINAL_FEATURE_DECISIONS.csv",
    PROVENANCE / "QDIST_Family_Evaluation_Workbook_v1_0.docx",
    PROVENANCE / "QDIST_V4_0_0_FREEZE_CONTRACT.md",
    PROVENANCE / "QDIST_Validation_Checklist_v1_0.csv",
    PROVENANCE / "QDIST_Ten_Domain_Dashboard_v1_0.csv",
    PROVENANCE / "QDIST_Gate_Summary_FINAL_v1_0.csv",
    PROVENANCE / "QDIST_V400_FINALIZATION_IMPLEMENTATION_REPORT.md",
]
for path in required_provenance:
    if not path.exists():
        raise FileNotFoundError(path)
print("Source candidate:", SOURCE_ROOT)
print("Final candidate:", FINAL_ROOT)

In [ ]:
if RUN_FINALIZATION:
    manifest = finalize_candidate(
        source_root=SOURCE_ROOT,
        final_root=FINAL_ROOT,
        scientific_review_decision=SCIENTIFIC_REVIEW_DECISION,
        scientific_reviewer=SCIENTIFIC_REVIEWER,
        scientific_review_rationale=(
            "Post-cohort G1-G10 scientific audit accepted with explicit sparse-positive, "
            "reliability, and AI-assisted event-review qualifications."
        ),
        provenance_files=required_provenance,
    )
    print(json.dumps(manifest, indent=2))

In [ ]:
required = {
    "measurement_version": "qdist-v4.0.0",
    "freeze_status": "ready_for_atomic_freeze",
    "freeze_allowed": True,
    "recording_count": 519,
    "participant_count": 224,
    "available_recording_count": 519,
    "positive_recording_count": 6,
    "valid_zero_recording_count": 513,
    "event_review_item_count": 60,
    "event_review_standardized_png_count": 60,
    "figure_count": 23,
    "main_figure_bundle_count": 15,
    "gallery_bundle_count": 8,
    "required_panels_complete": True,
    "panel_i_status": "APPLICABLE_complete_event_verification",
    "numerical_equivalence_to_cohort_candidate": True,
    "numerical_equivalence_to_qdist_v311": True,
    "feature_values_recomputed": False,
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
    "complete_nonlinear_distortion_claim_allowed": False,
    "missing_values_imputed": False,
}
for key, expected in required.items():
    if manifest.get(key) != expected:
        raise AssertionError(f"Manifest mismatch {key}: {manifest.get(key)!r}")
print("Final manifest contract verified.")

In [ ]:
print("QDIST v4.0.0 FINALIZATION COMPLETE")
print("Candidate is ready for atomic freeze.")
print("PUBLISH_AND_FREEZE remains", PUBLISH_AND_FREEZE)